# AI Inventory Intelligence & Supplier Orchestration 📦🛡️
Run the full Streamlit Dashboard directly from Google Colab!

**Instructions:**
1. Run the first cell to setup the environment.
2. Add your API keys in the second cell.
3. Run the final cell to launch. **If the iframe below is blank, use the 'Fallback URL' printed in the cell output.**

In [ ]:
!git clone https://github.com/lmudu2/ai-inventory-intelligence.git
%cd ai-inventory-intelligence
!pip install -r requirements.txt
!npm install localtunnel

In [ ]:
import os
# ⚠️ ENTER YOUR STAGING API KEYS HERE ⚠️
os.environ['GROQ_API_KEY'] = 'gsk_...'
os.environ['SENDGRID_API_KEY'] = 'SG...'
os.environ['SENDGRID_FROM_EMAIL'] = 'your_email@domain.com'
os.environ['SENDGRID_TO_EMAIL'] = 'target_email@domain.com'
os.environ['NEWSDATA_API_KEY'] = 'pub_...'

print("Environment Variables Set!")

In [ ]:
import time
import socket
from google.colab import output
import urllib
import subprocess

def wait_for_port(port, timeout=40):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                return False
            time.sleep(1)

print("🧹 Cleaning up existing processes...")
!fuser -k 8501/tcp

print("🚀 Launching AI Supply Chain Dashboard...")
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &")

print("⏳ Initializing (usually ~15s)... Check bottom of output for the Fallback URL if iframe is blank.")
if wait_for_port(8501):
    print("✅ Dashboard Backend Ready!")
    
    # Start Fallback Tunnel (using npx for reliability)
    print("\n--- 🔗 FALLBACK ACCESS (Use if iframe below is blank) ---")
    print("Tunnel Password (Endpoint IP):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
    # Launch tunnel and catch output
    subprocess.Popen(["npx", "localtunnel", "--port", "8501"], stdout=open('localtunnel.log', 'w'))
    time.sleep(5)
    try:
        with open('localtunnel.log', 'r') as f:
            print("Temporary URL:", f.read().split('url is: ')[-1])
    except:
        print("Localtunnel still starting... please wait a moment.")
    
    print("\n--- 🖥️ RENDERING IFRAME BELOW ---")
    output.serve_kernel_port_as_iframe(8501)
else:
    print("❌ Startup failed. Displaying logs:")
    !cat streamlit.log